# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Charger les données

In [ ]:
import json
import ast
from pathlib import Path

def load_pyomo_data(input_path="../data/Mercantile_data.json"):
    """Charge les donnees externes depuis un fichier JSON."""
    input_file = Path(input_path)

    with open(input_file, "r") as f:
        data = json.load(f)

    def _convert_key(key):
        if not isinstance(key, str):
            return key
        if key.startswith("(") and key.endswith(")"):
            try:
                return ast.literal_eval(key)
            except Exception:
                return key
        try:
            return int(key)
        except Exception:
            return key

    # Convertir les dictionnaires de parametres indexes
    params = data.get("params", {})
    for pname, pval in list(params.items()):
        if isinstance(pval, dict):
            params[pname] = {_convert_key(k): v for k, v in pval.items()}

    cartesian = data.get("cartesian_data", {})
    for cname, cval in list(cartesian.items()):
        if isinstance(cval, dict):
            cartesian[cname] = {_convert_key(k): v for k, v in cval.items()}

    data["params"] = params
    data["cartesian_data"] = cartesian
    return data

# Charger les données
data = load_pyomo_data()

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.MOIS = Set(initialize=data['sets']['MOIS'])
model.PERIODES = Set(initialize=data['sets']['PERIODES'])
model.ARC = Set(dimen=2, initialize=[(i0,i1) for i0 in model.MOIS for i1 in model.PERIODES])

## 🔹 Parameters

In [ ]:
model.ESPACE = Param(model.MOIS, initialize=data['params']['ESPACE'], within=NonNegativeReals)
model.COUT = Param(model.PERIODES, initialize=data['params']['COUT'], within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.MOIS, model.PERIODES, domain=NonNegativeReals)

## 🔹 Constraints

In [ ]:
model.c_for_0 = ConstraintList()
for m in model.MOIS:
    model.c_for_0.add(sum(sum(model.X[k, j] for j in model.PERIODES if k + j > m) for k in model.MOIS if k <= m) >= model.ESPACE[m])

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.COUT[p] * model.X[m, p] for (m,p) in model.ARC), sense=minimize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')